In [ ]:
import torch
from train import run_baseline_experiment, run_lora_experiment, run_lora_experiment_multiple_seeds
from utils import plot_results
print("CUDA available?", torch.cuda.is_available())
import lightning
print(lightning.__version__)
import numpy as np



In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
%%bash
python -c "import torch; print(torch.version.cuda)"
python -c "import torch; print(torch.cuda.device_count())"



In [ ]:
for i in range(torch.cuda.device_count()):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")

## FPFT

In [ ]:
#TODO: rewrite all launches for the new API
#TODO: make them rerunnable (calculate acc over several runs with fixed randomstates)

In [ ]:
# This either trains a new full‐parameter model on classes [0..4]
# or loads it from "base_model.ckpt" if found:
baseline_acc = run_baseline_experiment()
print("Baseline FPFT accuracy:", baseline_acc)

## LoRA

In [ ]:
# A:Gaussian; B:Zero
NUM_LAUNCHES = 20
seeds_to_try = np.arange(1,NUM_LAUNCHES+1)
average_acc = run_lora_experiment_multiple_seeds(seeds=seeds_to_try,rank=1, train_A=True, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=0, use_stochastic=False)
print("LoRA (A:Gaussian; B:Zero):", average_acc)

In [ ]:
# A:Zero; B:Gaussian
NUM_LAUNCHES = 20
seeds_to_try = np.arange(1,NUM_LAUNCHES+1)
average_acc = run_lora_experiment_multiple_seeds(seeds=seeds_to_try, rank=1, train_A=True, train_B=True, init_method_A='zero', init_method_B='gaussian', init_method_B='zero', merge_frequency=0, use_stochastic=False)
print("LoRA (A:Zero; B:Gaussian):", average_acc)

## COLA

In [ ]:
# A:Gaussian; B:Zero
acc = run_lora_experiment(rank=1, train_A=True, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=1)
print("COLA (A:Gaussian; B:Zero):", acc)

In [ ]:
# A:Zero; B:Gaussian

acc = run_lora_experiment(
    rank=1,
    train_A=True,               # Train both A and B
    train_B=True,
    init_method_A='zero',   # A ~ Gaussian
    init_method_B='gaussian',       # B ~ Zero
    merge_frequency=1           # or 1; see note below
)
print("COLA (A:Zero; B:Gaussian):", acc)

In [ ]:
# A:Gaussian; B:Gaussian
acc = run_lora_experiment(rank=1, train_A=True, train_B=True, init_method_A='gaussian', init_method_B='gaussian', merge_frequency=1)
print("COLA (A:Gaussian; B:Gaussian):", acc)

## AsymmLoRA

In [ ]:
# A:Gaussian; B:Zero
acc = run_lora_experiment(rank=1, train_A=False, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=0)
print("AsymmLoRA (A:Gaussian; B:Zero):", acc)

In [ ]:
# A:Zero; B:Gaussian
acc = run_lora_experiment(rank=1, train_A=True, train_B=False, init_method_A='zero', init_method_B='gaussian', merge_frequency=0)
print("AsymmLoRA (A:Gaussian; B:Zero):", acc)

## RAC-LoRA

In [ ]:
# A:Gaussian; B:Zero
acc = run_lora_experiment(rank=1, train_A=False, train_B=True, init_method_A='gaussian', init_method_B='zero', merge_frequency=1)
print("RAC-LoRA (A:Gaussian; B:Zero):", acc)

In [ ]:
# A:Zero; B:Gaussian
acc = run_lora_experiment(rank=1, train_A=True, train_B=False, init_method_A='zero', init_method_B='gaussian', merge_frequency=1)
print("RAC-LoRA (A:Gaussian; B:Zero):", acc)

## Bernoulli-LoRA

In [ ]:
# in progress, one needs to experiments with prob and merge_frequency

acc = run_lora_experiment(rank=1,
    train_A=True,
    train_B=False,
    init_method_A='zero',
    init_method_B='gaussian',
    merge_frequency=1,         
    use_stochastic=True,       
    prob=0.5,
    deterministic_init=True,
    init_train_zero=True,     
    gaussian_resample=True
)
print("Final test acc =", acc)
